In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# 1. Datan lataaminen
df = pd.read_csv('cicids2017_cleaned.csv')

# Käsitellään äärettömyydet ja puuttuvat arvot
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Valitaan 10 ominaisuutta
selected_features = [
    'Init_Win_bytes_forward', 'Flow IAT Mean', 'Packet Length Std', 'Subflow Fwd Bytes',
    'Flow Duration', 'Bwd Packet Length Mean', 'Total Length of Fwd Packets',
    'PSH Flag Count', 'Flow Packets/s', 'Destination Port'
]
label = 'Attack Type'

# 2. Mallien määrittely
# Määritellään vain muuttuvat osat: algoritmi ja sen omat parametrit
# SVM mallin kouluttaminen 20 000 rivin aineistolla kesti 4 minuuttia.
# Ajanpuutteen vuoksi rivejä ei lisätty, vaikka se todennäköisesti parantaisi mallin suorituskykyä.
models = {
    'SVM': (20000, SVC(cache_size=2000), {
        'pca__n_components': [8],
        'model__C': [1, 10],
        'model__kernel': ['rbf'],
        'smote__k_neighbors': [1, 3]
    }),
    'Naive Bayes': (100000, GaussianNB(), {
        'pca__n_components': [5, 8],
        'model__var_smoothing': [1e-9, 1e-8],
        'smote__k_neighbors': [1]
    })
}

# 3. Mallien harjoittaminen silmukassa
for name, (size, algo, params) in models.items():
    print(f"\n--- Ajetaan {name} (otos: {size}) ---")

    # Otos ja harvinaisten luokkien poisto (väh. 15 kpl)
    df_s = df[selected_features + [label]].sample(n=size, random_state=42)
    v_counts = df_s[label].value_counts()
    df_s = df_s[df_s[label].isin(v_counts[v_counts >= 15].index)]

    X = df_s.drop(label, axis=1)
    y = df_s[label]

    # Datan jako 60/20/20
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    _, X_test, _, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Rakennetaan putki (yhteiset osat + vaihtuva algoritmi)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(random_state=42)),
        ('smote', SMOTE(random_state=42)),
        ('model', algo)
    ])

    # Suoritetaan Grid Search
    grid = GridSearchCV(pipe, params, cv=3, n_jobs=-1, error_score=0, verbose=1)
    grid.fit(X_train, y_train)

    # Tulostetaan tulokset
    y_pred = grid.predict(X_test)
    print(f"parhaat: {grid.best_params_}")
    print(f"tarkkuus: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))


--- Ajetaan SVM (otos: 20000) ---
Fitting 3 folds for each of 4 candidates, totalling 12 fits
parhaat: {'model__C': 10, 'model__kernel': 'rbf', 'pca__n_components': 8, 'smote__k_neighbors': 1}
tarkkuus: 0.8824
                precision    recall  f1-score   support

          Bots       0.02      1.00      0.03         3
   Brute Force       0.13      1.00      0.22        15
          DDoS       0.70      1.00      0.82       210
           DoS       0.94      0.88      0.91       301
Normal Traffic       1.00      0.87      0.93      3336
 Port Scanning       0.67      0.99      0.80       133

      accuracy                           0.88      3998
     macro avg       0.57      0.96      0.62      3998
  weighted avg       0.96      0.88      0.91      3998


--- Ajetaan Naive Bayes (otos: 100000) ---
Fitting 3 folds for each of 4 candidates, totalling 12 fits
parhaat: {'model__var_smoothing': 1e-09, 'pca__n_components': 8, 'smote__k_neighbors': 1}
tarkkuus: 0.6910
               